Import a úprava dat

In [1]:
import pandas as pd
import os

# Cesta ke složce s daty
data_folder = "Data"

# Pokud chceš otestovat úplně VŠECHNY CSV soubory ve složce najednou:
all_csv_files = [f for f in os.listdir(data_folder) if f.endswith('.csv')]

print(f"--- Zahájení kontroly datových souborů (Celkem nalezeno: {len(all_csv_files)}) ---\n")

summary = []

for file_name in all_csv_files:
    file_path = os.path.join(data_folder, file_name)
    try:
        # Zkusíme načíst soubor (předpokládáme středník a utf-8, co jsme řešili minule)
        # Pokud by to házelo chybu u chřipky z roku 1994, zkus změnit na encoding='cp1250'
        df = pd.read_csv(file_path, sep=';', encoding='utf-8')
        
        row_count = len(df)
        col_count = len(df.columns)
        first_col = df.columns[0]
        
        summary.append({
            "Soubor": file_name,
            "Stav": "✅ OK",
            "Řádků": row_count,
            "Sloupců": col_count,
            "První sloupec": first_col
        })
    except Exception as e:
        summary.append({
            "Soubor": file_name,
            "Stav": f"❌ CHYBA: {str(e)[:50]}...",
            "Řádků": 0,
            "Sloupců": 0,
            "První sloupec": "-"
        })

# Převedení výsledků do tabulky pro lepší přehlednost
report_df = pd.DataFrame(summary)
display(report_df)

print("\n--- Kontrola dokončena ---")

--- Zahájení kontroly datových souborů (Celkem nalezeno: 23) ---



,Soubor,Stav,Řádků,Sloupců,První sloupec
0,Hospitalizace_umrti_chripka(1994-2024).csv,✅ OK,17763,9,rok
1,Hospitalizace_umrti_Covid-19(2020-2025)_CR.csv,✅ OK,2136,124,Datum - ČR
2,Hospitalizace_umrti_Covid-19(2020-2025)_HKK.csv,✅ OK,1816,124,Datum - Královéhradecký kraj
3,Hospitalizace_umrti_Covid-19(2020-2025)_HMP.csv,✅ OK,2066,124,Datum - Hlavní město Praha
4,Hospitalizace_umrti_Covid-19(2020-2025)_JHC.csv,✅ OK,1794,124,Datum - Jihočeský kraj
5,Hospitalizace_umrti_Covid-19(2020-2025)_JMK.csv,✅ OK,2009,124,Datum - Jihomoravský kraj
6,Hospitalizace_umrti_Covid-19(2020-2025)_KVK.csv,✅ OK,1587,124,Datum - Karlovarský kraj
7,Hospitalizace_umrti_Covid-19(2020-2025)_LBK.csv,✅ OK,1660,124,Datum - Liberecký kraj
8,Hospitalizace_umrti_Covid-19(2020-2025)_MSK.csv,✅ OK,2040,124,Datum - Moravskoslezský kraj
9,Hospitalizace_umrti_Covid-19(2020-2025)_OLK.csv,✅ OK,1768,124,Datum - Olomoucký kraj



--- Kontrola dokončena ---


In [2]:
import pandas as pd
import glob
import os

data_folder = "Data"

# --- 1. POMOCNÉ FUNKCE A ČÍSELNÍKY PRO PŘEVOD OKRES -> KRAJ ---

# Pomocný slovník pro převod zkratek ze souborů na NUTS kód a celý název
mapovani_zkratek = {
    'HMP': ('CZ010', 'Hlavní město Praha'),
    'STC': ('CZ020', 'Středočeský kraj'),
    'JHC': ('CZ031', 'Jihočeský kraj'),
    'PLK': ('CZ032', 'Plzeňský kraj'),
    'KVK': ('CZ041', 'Karlovarský kraj'),
    'ULK': ('CZ042', 'Ústecký kraj'),
    'LBK': ('CZ051', 'Liberecký kraj'),
    'HKK': ('CZ052', 'Královéhradecký kraj'),
    'PAK': ('CZ053', 'Pardubický kraj'),
    'VYS': ('CZ063', 'Kraj Vysočina'),
    'JMK': ('CZ064', 'Jihomoravský kraj'),
    'OLK': ('CZ071', 'Olomoucký kraj'),
    'ZLK': ('CZ072', 'Zlínský kraj'),
    'MSK': ('CZ080', 'Moravskoslezský kraj')
}

# Číselník názvů okresů (pro očkování)
okres_to_kraj_full = {
    'Benešov': 'Středočeský kraj', 'Beroun': 'Středočeský kraj', 'Kladno': 'Středočeský kraj',
    'Kolín': 'Středočeský kraj', 'Kutná Hora': 'Středočeský kraj', 'Mělník': 'Středočeský kraj', 
    'Mladá Boleslav': 'Středočeský kraj', 'Nymburk': 'Středočeský kraj', 'Praha-východ': 'Středočeský kraj', 
    'Praha-západ': 'Středočeský kraj', 'Příbram': 'Středočeský kraj', 'Rakovník': 'Středočeský kraj',
    'České Budějovice': 'Jihočeský kraj', 'Český Krumlov': 'Jihočeský kraj', 'Jindřichův Hradec': 'Jihočeský kraj',
    'Písek': 'Jihočeský kraj', 'Prachatice': 'Jihočeský kraj', 'Strakonice': 'Jihočeský kraj', 
    'Tábor': 'Jihočeský kraj', 'Plzeň-město': 'Plzeňský kraj', 'Plzeň-jih': 'Plzeňský kraj', 
    'Plzeň-sever': 'Plzeňský kraj', 'Domažlice': 'Plzeňský kraj', 'Klatovy': 'Plzeňský kraj',
    'Rokycany': 'Plzeňský kraj', 'Tachov': 'Plzeňský kraj', 'Cheb': 'Karlovarský kraj',
    'Karlovy Vary': 'Karlovarský kraj', 'Sokolov': 'Karlovarský kraj', 'Děčín': 'Ústecký kraj', 
    'Chomutov': 'Ústecký kraj', 'Litoměřice': 'Ústecký kraj', 'Louny': 'Ústecký kraj',
    'Most': 'Ústecký kraj', 'Teplice': 'Ústecký kraj', 'Ústí nad Labem': 'Ústecký kraj',
    'Česká Lípa': 'Liberecký kraj', 'Jablonec nad Nisou': 'Liberecký kraj', 'Liberec': 'Liberecký kraj',
    'Semily': 'Liberecký kraj', 'Hradec Králové': 'Královéhradecký kraj', 'Jičín': 'Královéhradecký kraj',
    'Náchod': 'Královéhradecký kraj', 'Rychnov nad Kněžnou': 'Královéhradecký kraj',
    'Trutnov': 'Královéhradecký kraj', 'Chrudim': 'Pardubický kraj', 'Pardubice': 'Pardubický kraj',
    'Svitavy': 'Pardubický kraj', 'Ústí nad Orlicí': 'Pardubický kraj', 'Havlíčkův Brod': 'Kraj Vysočina', 
    'Jihlava': 'Kraj Vysočina', 'Pelhřimov': 'Kraj Vysočina', 'Třebíč': 'Kraj Vysočina',
    'Žďár nad Sázavou': 'Kraj Vysočina', 'Blansko': 'Jihomoravský kraj', 'Brno-město': 'Jihomoravský kraj', 
    'Brno-venkov': 'Jihomoravský kraj', 'Břeclav': 'Jihomoravský kraj', 'Hodonín': 'Jihomoravský kraj',
    'Vyškov': 'Jihomoravský kraj', 'Znojmo': 'Jihomoravský kraj', 'Jeseník': 'Olomoucký kraj',
    'Olomouc': 'Olomoucký kraj', 'Prostějov': 'Olomoucký kraj', 'Přerov': 'Olomoucký kraj',
    'Šumperk': 'Olomoucký kraj', 'Kroměříž': 'Zlínský kraj', 'Uherské Hradiště': 'Zlínský kraj',
    'Vsetín': 'Zlínský kraj', 'Zlín': 'Zlínský kraj', 'Bruntál': 'Moravskoslezský kraj',
    'Frýdek-Místek': 'Moravskoslezský kraj', 'Karviná': 'Moravskoslezský kraj', 'Nový Jičín': 'Moravskoslezský kraj',
    'Opava': 'Moravskoslezský kraj', 'Ostrava-město': 'Moravskoslezský kraj'
}

# Číselník kódů okresů (pro detailní úmrtí)
kod_kraje_to_nazev = {
    'CZ010': 'Hlavní město Praha', 'CZ020': 'Středočeský kraj', 'CZ031': 'Jihočeský kraj',
    'CZ032': 'Plzeňský kraj', 'CZ041': 'Karlovarský kraj', 'CZ042': 'Ústecký kraj',
    'CZ051': 'Liberecký kraj', 'CZ052': 'Královéhradecký kraj', 'CZ053': 'Pardubický kraj',
    'CZ063': 'Kraj Vysočina', 'CZ064': 'Jihomoravský kraj', 'CZ071': 'Olomoucký kraj',
    'CZ072': 'Zlínský kraj', 'CZ080': 'Moravskoslezský kraj'
}

nazev_to_kod = {v: k for k, v in kod_kraje_to_nazev.items()}

def agreguj_chripku_na_kraje(df):
    # 1. Přiřazení celého názvu kraje podle okresu (používá tvůj okres_to_kraj_full)
    df['Kraj_Název'] = df['uzemni_celek'].map(okres_to_kraj_full)
    
    # 2. Vyčištění od nenamapovaných řádků
    df_clean = df.dropna(subset=['Kraj_Název']).copy()
    
    # 3. Agregace dat (sumarizace za kraj a sezónu)
    df_agg = df_clean.groupby(['Kraj_Název', 'sezona']).agg({
        'pocet_vakcinovanych': 'sum',
        'demografie': 'sum'
    }).reset_index()
    
    # 4. DOPLNĚNÍ Kraj_ID s využitím tvého číselníku kod_kraje_to_nazev
    # Prohodíme klíče a hodnoty: {'Název': 'CZxxx'}
    df_agg.insert(0, 'Kraj_ID', df_agg['Kraj_Název'].map(nazev_to_kod))
    
    # 5. Výpočet proočkovanosti v %
    df_agg['proockovanost_procenta'] = (df_agg['pocet_vakcinovanych'] / df_agg['demografie']) * 100
    
    return df_agg

def preved_detailni_umrti_na_kraje(df):
    df_new = df.copy()
    # Extrakce kódu kraje z kódu okresu (CZ0531 -> CZ053)
    df_new['Kod_Kraje'] = df_new['okres_bydliste'].str[:5]
    df_new['Kraj_Název'] = df_new['Kod_Kraje'].map(kod_kraje_to_nazev)
    return df_new

def dopln_id_podle_nazvu(df):
    if 'Kraj_ID' not in df.columns:
        df.insert(0, 'Kraj_ID', df['Kraj'].map(nazev_to_kod))
    return df

# --- 2. SLOUČENÁ KRAJSKÁ DATA COVID-19 ---
kraj_files = glob.glob(os.path.join(data_folder, "Hospitalizace_umrti_Covid-19(2020-2025)_*.csv"))
kraj_files = [f for f in kraj_files if "_CR.csv" not in f]

seznam_df = []

for file in kraj_files:
    temp_df = pd.read_csv(file, sep=';', encoding='utf-8')
    
    # Sjednocení prvního sloupce na 'Datum'
    temp_df.rename(columns={temp_df.columns[0]: 'Datum'}, inplace=True)
    
    # Získání zkratky z názvu souboru (např. 'HKK')
    zkratka = os.path.basename(file).split('_')[-1].replace('.csv', '')
    
    # Získání NUTS kódu a celého názvu ze slovníku
    nuts_kod, plny_nazev = mapovani_zkratek.get(zkratka, (zkratka, "Neznámý kraj"))
    
    # Vložení nových identifikačních sloupců na začátek tabulky
    temp_df.insert(0, 'Kraj_Název', plny_nazev)
    temp_df.insert(0, 'Kraj_ID', nuts_kod) # NUTS kód (CZxxx)
    
    seznam_df.append(temp_df)

df_covid_umrti_hosp_kraje = pd.concat(seznam_df, ignore_index=True)
df_covid_umrti_hosp_kraje['Datum'] = pd.to_datetime(df_covid_umrti_hosp_kraje['Datum'], dayfirst=True)

# --- 3. COVID-19: CELOREPUBLIKOVÁ DATA ---
df_covid_umrti_hosp_cr = pd.read_csv(os.path.join(data_folder, "Hospitalizace_umrti_Covid-19(2020-2025)_CR.csv"), sep=';', encoding='utf-8')
df_covid_umrti_hosp_cr.rename(columns={df_covid_umrti_hosp_cr.columns[0]: 'Datum'}, inplace=True)
df_covid_umrti_hosp_cr['Datum'] = pd.to_datetime(df_covid_umrti_hosp_cr['Datum'], dayfirst=True)

# --- 4. OČKOVÁNÍ: COVID-19 ---
df_covid_ockov_kraje = pd.read_csv(os.path.join(data_folder, "Ockovani_kraj_Covid-19(2020-2025).csv"), sep=';', encoding='utf-8')

# --- 5. CHŘIPKA: HOSPITALIZACE A ÚMRTÍ (HISTORIE) ---
df_chripka_umrti_hosp_kraje = pd.read_csv(os.path.join(data_folder, "Hospitalizace_umrti_chripka(1994-2024).csv"), sep=';', encoding='utf-8')

# --- 6. CHŘIPKA: ÚMRTÍ (DETAILNÍ DATA 2015-2024 + TRANSFORMACE) ---
df_raw_umrti_detail = pd.read_csv(os.path.join(data_folder, "Umrti_chripka(2015-2024).csv"), sep=';', encoding='utf-8')
df_chripka_umrti_kraje = preved_detailni_umrti_na_kraje(df_raw_umrti_detail)

# --- 7. CHŘIPKA: KRAJSKÉ SROVNÁNÍ ÚMRTÍ (ABS, NA 100tis, PROCENTA) ---
df_chripka_umrti_abs_kraje = pd.read_csv(os.path.join(data_folder, "Umrti_chripka_kraje_abs(1994-2023).csv"), sep=';', encoding='utf-8')
df_chripka_umrti_norm_kraje = pd.read_csv(os.path.join(data_folder, "Umrti_chripka_kraje_na100tis(1994-2023).csv"), sep=';', encoding='utf-8')
df_chripka_umrti_perc_kraje = pd.read_csv(os.path.join(data_folder, "Umrti_chripka_kraje_procenta(1994-2023).csv"), sep=';', encoding='utf-8')

df_chripka_umrti_abs_kraje = dopln_id_podle_nazvu(df_chripka_umrti_abs_kraje)
df_chripka_umrti_norm_kraje = dopln_id_podle_nazvu(df_chripka_umrti_norm_kraje)
df_chripka_umrti_perc_kraje = dopln_id_podle_nazvu(df_chripka_umrti_perc_kraje)

# --- 8. OČKOVÁNÍ: CHŘIPKA (Transformace z okresů na celé názvy krajů) ---
df_raw_65 = pd.read_csv(os.path.join(data_folder, "Ockovani_chripka(2010-2025)_65plus.csv"), sep=';', encoding='utf-8')
df_raw_vsichni = pd.read_csv(os.path.join(data_folder, "Ockovani_chripka(2010-2025)_vsichni.csv"), sep=';', encoding='utf-8')

df_chripka_ockov_65plus_kraje = agreguj_chripku_na_kraje(df_raw_65)
df_chripka_ockov_vsichni_kraje = agreguj_chripku_na_kraje(df_raw_vsichni)

print("✅ Všech 10 datových celků bylo úspěšně načteno a sjednoceno na úroveň krajů.")

✅ Všech 10 datových celků bylo úspěšně načteno a sjednoceno na úroveň krajů.


In [3]:
# --- KONTROLNÍ BLOK IMPORTU ---

# Seznam proměnných a jejich popisů pro report
datasets = {
    "df_covid_umrti_hosp_kraje": "COVID-19 - Sloučené kraje (14 souborů)",
    "df_covid_umrti_hosp_cr": "COVID-19 - Celá ČR",
    "df_covid_ockov_kraje": "Očkování COVID-19 - Krajské",
    "df_chripka_umrti_hosp_kraje": "Chřipka - Historické hosp. (1994-2024)",
    "df_chripka_umrti_kraje": "Chřipka - Detailní úmrtí (2015-2024)",
    "df_chripka_umrti_abs_kraje": "Chřipka - Úmrtí kraje (Absolutně)",
    "df_chripka_umrti_norm_kraje": "Chřipka - Úmrtí kraje (na 100 tis.)",
    "df_chripka_umrti_perc_kraje": "Chřipka - Úmrtí kraje (Procenta)",
    "df_chripka_ockov_65plus_kraje": "Očkování chřipka - 65+",
    "df_chripka_ockov_vsichni_kraje": "Očkování chřipka - Všichni"
}

check_results = []

for var_name, description in datasets.items():
    try:
        # Získáme objekt proměnné podle jejího názvu
        obj = globals().get(var_name)
        
        if obj is not None:
            check_results.append({
                "Proměnná": var_name,
                "Popis dat": description,
                "Stav": "✅ Načteno",
                "Počet řádků": len(obj),
                "Počet sloupců": len(obj.columns)
            })
        else:
            check_results.append({
                "Proměnná": var_name,
                "Popis dat": description,
                "Stav": "❌ Chybí (nenalezeno)",
                "Počet řádků": 0,
                "Počet sloupců": 0
            })
    except Exception as e:
        check_results.append({
            "Proměnná": var_name,
            "Popis dat": description,
            "Stav": f"⚠️ Chyba: {str(e)[:30]}",
            "Počet řádků": 0,
            "Počet sloupců": 0
        })

# Zobrazení výsledného reportu
final_report = pd.DataFrame(check_results)
print("--- FINÁLNÍ KONTROLA DATOVÝCH SAD V PAMĚTI ---")
display(final_report)

# Malý test kontinuity datumu u hlavního datasetu
if 'df_covid_umrti_hosp_kraje' in globals():
    min_date = df_covid_umrti_hosp_kraje['Datum'].min().strftime('%d.%m.%Y')
    max_date = df_covid_umrti_hosp_kraje['Datum'].max().strftime('%d.%m.%Y')
    print(f"\n📅 Časový rozsah COVID-19 (kraje): {min_date} až {max_date}")

--- FINÁLNÍ KONTROLA DATOVÝCH SAD V PAMĚTI ---


,Proměnná,Popis dat,Stav,Počet řádků,Počet sloupců
0,df_covid_umrti_hosp_kraje,COVID-19 - Sloučené kraje (14 souborů),✅ Načteno,25402,126
1,df_covid_umrti_hosp_cr,COVID-19 - Celá ČR,✅ Načteno,2136,124
2,df_covid_ockov_kraje,Očkování COVID-19 - Krajské,✅ Načteno,357518,9
3,df_chripka_umrti_hosp_kraje,Chřipka - Historické hosp. (1994-2024),✅ Načteno,17763,9
4,df_chripka_umrti_kraje,Chřipka - Detailní úmrtí (2015-2024),✅ Načteno,3295,11
5,df_chripka_umrti_abs_kraje,Chřipka - Úmrtí kraje (Absolutně),✅ Načteno,103,33
6,df_chripka_umrti_norm_kraje,Chřipka - Úmrtí kraje (na 100 tis.),✅ Načteno,103,33
7,df_chripka_umrti_perc_kraje,Chřipka - Úmrtí kraje (Procenta),✅ Načteno,103,33
8,df_chripka_ockov_65plus_kraje,Očkování chřipka - 65+,✅ Načteno,195,6
9,df_chripka_ockov_vsichni_kraje,Očkování chřipka - Všichni,✅ Načteno,195,6



📅 Časový rozsah COVID-19 (kraje): 11.03.2020 až 13.01.2026


In [13]:
# Seznam vašich finálních datasetů pro kontrolu
datasets = {
    "COVID: Hospitalizace a úmrtí (kraje)": df_covid_umrti_hosp_kraje,
    "COVID: Očkování (kraje)": df_covid_ockov_kraje,
    "Chřipka: Hospitalizace a úmrtí (historie)": df_chripka_umrti_hosp_kraje,
    "Chřipka: Úmrtí (detailní)": df_chripka_umrti_kraje,
    "Chřipka: Úmrtí ABS (historie)": df_chripka_umrti_abs_kraje,
    "Chřipka: Úmrtí na 100tis": df_chripka_umrti_norm_kraje,
    "Chřipka: Úmrtí %": df_chripka_umrti_perc_kraje,
    "Chřipka: Očkování 65+": df_chripka_ockov_65plus_kraje,
    "Chřipka: Očkování všichni": df_chripka_ockov_vsichni_kraje,
    "COVID: Celorepubliková data": df_covid_umrti_hosp_cr
}

for nazev, df in datasets.items():
    print(f"\n--- {nazev} ---")
    # Zobrazíme jen první 2 řádky pro přehlednost
    display(df.iloc[50:65])


--- COVID: Hospitalizace a úmrtí (kraje) ---


,Kraj_ID,Kraj_Název,Datum,Celkový počet nakažených,Počet hospitalizovaných celkem v daném dni,Počet hospitalizovaných na JIP celkem v daném dni,Počet nově hospitalizovaných celkem,Počet nově hospitalizovaných na JIP,Počet zemřelých,Počet antigenních testů,...,Zemřelí ve věku 45-49 let,Zemřelí ve věku 50-54 let,Zemřelí ve věku 55-59 let,Zemřelí ve věku 60-64 let,Zemřelí ve věku 65-69 let,Zemřelí ve věku 70-74 let,Zemřelí ve věku 75-79 let,Zemřelí ve věku 80-84 let,Zemřelí ve věku 85+ let,Zemřelí - věk neznámý
50,CZ052,Královéhradecký kraj,2020-05-12,2.0,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
51,CZ052,Královéhradecký kraj,2020-05-13,NaN,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
52,CZ052,Královéhradecký kraj,2020-05-14,1.0,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
53,CZ052,Královéhradecký kraj,2020-05-15,1.0,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
54,CZ052,Královéhradecký kraj,2020-05-16,NaN,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
55,CZ052,Královéhradecký kraj,2020-05-17,NaN,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
56,CZ052,Královéhradecký kraj,2020-05-18,NaN,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
57,CZ052,Královéhradecký kraj,2020-05-19,1.0,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58,CZ052,Královéhradecký kraj,2020-05-20,1.0,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
59,CZ052,Královéhradecký kraj,2020-05-21,NaN,1.0,1.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



--- COVID: Očkování (kraje) ---


,id,datum,vakcina,kraj_nuts_kod,kraj_nazev,vekova_skupina,prvnich_davek,druhych_davek,celkem_davek
50,3faa0eec-be57-45ea-a6c7-efbf7984255a,28.12.2020,Comirnaty,CZ064,Jihomoravský kraj,40-44,120,0,120
51,eb04ac58-c65d-4318-95b9-1f1007257c2e,28.12.2020,Comirnaty,CZ064,Jihomoravský kraj,25-29,72,0,72
52,5ab57c3f-05c7-469e-b1a6-fb968a74cb5e,28.12.2020,Comirnaty,CZ010,Hlavní město Praha,60-64,99,0,99
53,b4890fad-e4b2-44d1-a7c7-31664cef3495,28.12.2020,Comirnaty,CZ064,Jihomoravský kraj,18-24,9,0,9
54,5d7e3b14-4001-4dc7-a968-49fb2f6eb72c,28.12.2020,Comirnaty,CZ010,Hlavní město Praha,50-54,144,0,144
55,64d48b7c-13cf-45c3-947a-95c2969273d0,28.12.2020,Comirnaty,CZ064,Jihomoravský kraj,75-79,11,0,11
56,f734116c-0381-4fc8-8c60-7528baee2655,28.12.2020,Comirnaty,CZ064,Jihomoravský kraj,35-39,108,0,108
57,5d06c970-779b-4030-994a-de9a2a2ee7bb,28.12.2020,Comirnaty,CZ010,Hlavní město Praha,18-24,41,0,41
58,2a641e67-c89d-425b-9368-1537d8d21ae2,28.12.2020,Comirnaty,CZ010,Hlavní město Praha,55-59,120,0,120
59,a1a0adf4-ea06-4666-9296-b2e9280152a9,28.12.2020,Comirnaty,CZ010,Hlavní město Praha,70-74,79,0,79



--- Chřipka: Hospitalizace a úmrtí (historie) ---


...

,Kraj_ID,Kraj_Název,sezona,pocet_vakcinovanych,demografie,proockovanost_procenta
50,CZ063,Kraj Vysočina,2015-2016,18766,95262.0,19.699355
51,CZ063,Kraj Vysočina,2016-2017,20379,97958.0,20.803814
52,CZ063,Kraj Vysočina,2017-2018,21328,100357.0,21.252130
53,CZ063,Kraj Vysočina,2018-2019,23356,102513.0,22.783452
54,CZ063,Kraj Vysočina,2019-2020,25015,104291.0,23.985771
55,CZ063,Kraj Vysočina,2020-2021,25703,105748.0,24.305897
56,CZ063,Kraj Vysočina,2021-2022,27702,107024.0,25.883914
57,CZ063,Kraj Vysočina,2022-2023,27271,108999.0,25.019496
58,CZ063,Kraj Vysočina,2023-2024,29060,110988.0,26.183011
59,CZ063,Kraj Vysočina,2024-2025,28310,112286.0,25.212404



--- Chřipka: Očkování všichni ---


,Kraj_ID,Kraj_Název,sezona,pocet_vakcinovanych,demografie,proockovanost_procenta
50,CZ063,Kraj Vysočina,2015-2016,23225,509475.0,4.558614
51,CZ063,Kraj Vysočina,2016-2017,25101,508952.0,4.931899
52,CZ063,Kraj Vysočina,2017-2018,26191,508916.0,5.146429
53,CZ063,Kraj Vysočina,2018-2019,28285,509274.0,5.553985
54,CZ063,Kraj Vysočina,2019-2020,30094,509813.0,5.902949
55,CZ063,Kraj Vysočina,2020-2021,30893,508852.0,6.071117
56,CZ063,Kraj Vysočina,2021-2022,33269,504025.0,6.600665
57,CZ063,Kraj Vysočina,2022-2023,32977,514777.0,6.406075
58,CZ063,Kraj Vysočina,2023-2024,39147,517960.0,7.557920
59,CZ063,Kraj Vysočina,2024-2025,38666,517647.0,7.469569



--- COVID: Celorepubliková data ---


,Datum,Celkový počet nakažených,Počet hospitalizovaných celkem v daném dni,Počet hospitalizovaných na JIP celkem v daném dni,Počet nově hospitalizovaných celkem,Počet nově hospitalizovaných na JIP,Počet zemřelých,Počet antigenních testů,Počet PCR testů,počet testů CELKEM,...,Zemřelí ve věku 45-49 let,Zemřelí ve věku 50-54 let,Zemřelí ve věku 55-59 let,Zemřelí ve věku 60-64 let,Zemřelí ve věku 65-69 let,Zemřelí ve věku 70-74 let,Zemřelí ve věku 75-79 let,Zemřelí ve věku 80-84 let,Zemřelí ve věku 85+ let,Zemřelí - věk neznámý
50,2020-04-30,109.0,273.0,52.0,19.0,1.0,8.0,NaN,NaN,NaN,...,0.0,1.0,0.0,1.0,0.0,1.0,1.0,3.0,1.0,0.0
51,2020-05-01,57.0,244.0,46.0,9.0,1.0,8.0,NaN,NaN,NaN,...,0.0,0.0,0.0,1.0,0.0,0.0,2.0,3.0,2.0,0.0
52,2020-05-02,18.0,233.0,47.0,11.0,4.0,1.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
53,2020-05-03,26.0,232.0,45.0,6.0,0.0,3.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0
54,2020-05-04,39.0,236.0,47.0,18.0,5.0,5.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,2.0,0.0,1.0,0.0,2.0,0.0
55,2020-05-05,77.0,223.0,37.0,12.0,1.0,3.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0
56,2020-05-06,79.0,209.0,35.0,6.0,1.0,3.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0
57,2020-05-07,60.0,191.0,36.0,11.0,6.0,4.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,1.0,0.0
58,2020-05-08,46.0,185.0,40.0,11.0,7.0,5.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,2.0,0.0
59,2020-05-09,18.0,173.0,34.0,6.0,0.0,3.0,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0


Analýza - Prvotní přehled

In [5]:
import pandas as pd
import plotly.graph_objects as go

# --- 1. PŘÍPRAVA DAT (AGREGACE NA ROKY) ---

# COVID-19: Převod data na rok a suma
df_covid_umrti_hosp_cr['Datum'] = pd.to_datetime(df_covid_umrti_hosp_cr['Datum'])

# Vytvoříme pomocný sloupec 'rok' přímo v dataframe, aby s ním groupby mohl pracovat jako se sloupcem
df_covid_rocni = df_covid_umrti_hosp_cr.assign(rok=df_covid_umrti_hosp_cr['Datum'].dt.year).groupby('rok').agg({
    'Celkový počet nakažených': 'sum',
    'Počet zemřelých': 'sum'
}).reset_index()

# Chřipka: Suma za celou ČR podle roků
df_chripka_rocni = df_chripka_umrti_hosp_kraje.groupby('rok').agg({
    'pocet_hosp': 'sum',
    'umrti': 'sum'
}).reset_index()

# --- 2. DEFINICE JEDNOTNÉHO STYLU ---
# Definujeme barvy, aby byly v obou grafech stejné
colors = {'hosp': 'royalblue', 'umrti': 'black'}

def create_combined_chart(df, title, hosp_col, umrti_col):
    fig = go.Figure()
    
    # Sloupec pro hospitalizace
    fig.add_trace(go.Bar(
        x=df['rok'],
        y=df[hosp_col],
        name='Hospitalizace',
        marker_color=colors['hosp'],
        opacity=0.7
    ))
    
    # Sloupec pro úmrtí
    fig.add_trace(go.Scatter(
        x=df['rok'],
        y=df[umrti_col],
        name='Úmrtí',
        mode='lines+markers', # Čára i body pro každý rok
        line=dict(color=colors['umrti'], width=3),
        marker=dict(size=8)
    ))
    
    fig.update_layout(
        title=title,
        xaxis_title='Rok',
        yaxis_title='Počet osob (kumulativně za rok)',
        barmode='group',
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified"
    )
    return fig

# --- 3. VYKRESLENÍ GRAFŮ ---

# Graf pro COVID-19
fig_covid = create_combined_chart(df_covid_rocni, 'COVID-19: Roční přehled hospitalizací a úmrtí', 'Celkový počet nakažených', 'Počet zemřelých')
fig_covid.show()

# Graf pro Chřipku
fig_chripka = create_combined_chart(df_chripka_rocni, 'Chřipka: Roční přehled hospitalizací a úmrtí (Historie)', 'pocet_hosp', 'umrti')
fig_chripka.show()

In [17]:
# Příprava dat pro testy
df_testy_rocni = df_covid_umrti_hosp_cr.assign(
    rok=pd.to_datetime(df_covid_umrti_hosp_cr['Datum']).dt.year
).groupby('rok').agg({
    'Počet antigenních testů': 'sum',
    'Počet PCR testů': 'sum'
}).reset_index()

fig_testy = go.Figure()
fig_testy.add_trace(go.Bar(x=df_testy_rocni['rok'], y=df_testy_rocni['Počet PCR testů'], name='PCR testy', marker_color='darkblue'))
fig_testy.add_trace(go.Bar(x=df_testy_rocni['rok'], y=df_testy_rocni['Počet antigenních testů'], name='Antigenní testy', marker_color='lightblue'))

fig_testy.update_layout(
    title='Roční přehled diagnostiky COVID-19 v ČR',
    xaxis_title='Rok',
    yaxis_title='Počet provedených testů',
    barmode='stack', # Skládaný graf ukáže celkový objem i podíl typů
    template='plotly_white',
    hovermode="x unified"
)
fig_testy.show()

In [35]:
import pandas as pd
import plotly.express as px

# Sjednocení sloupců COVIDu do skupin chřipky
mapping_covid = {
    '0–19': ['Zemřelí ve věku 0-4 let', 'Zemřelí ve věku 5-9 let', 'Zemřelí ve věku 10-14 let', 'Zemřelí ve věku 15-19 let'],
    '20–44': [f'Zemřelí ve věku {i}-{i+4} let' for i in range(20, 45, 5)],
    '45–64': [f'Zemřelí ve věku {i}-{i+4} let' for i in range(45, 65, 5)],
    '65–74': ['Zemřelí ve věku 65-69 let', 'Zemřelí ve věku 70-74 let'],
    '75–84': ['Zemřelí ve věku 75-79 let', 'Zemřelí ve věku 80-84 let'],
    '85+': ['Zemřelí ve věku 85+ let']
}

df_covid_prep = df_covid_umrti_hosp_cr.copy()
df_covid_prep['rok'] = pd.to_datetime(df_covid_prep['Datum']).dt.year

covid_age_data = []
for skupina, sloupce in mapping_covid.items():
    existujici = [c for c in sloupce if c in df_covid_prep.columns]
    temp = df_covid_prep.groupby('rok')[existujici].sum().sum(axis=1).reset_index()
    temp.columns = ['rok', 'pocet_umrti']
    temp['vekova_skupina'] = skupina
    covid_age_data.append(temp)

df_covid_age_total = pd.concat(covid_age_data)

# Graf pro COVID-19
fig_covid_age = px.bar(
    df_covid_age_total, 
    x='rok', 
    y='pocet_umrti', 
    color='vekova_skupina',
    title='COVID-19: Vývoj úmrtnosti dle věkových skupin (2020–2026)',
    labels={'pocet_umrti': 'Počet úmrtí', 'rok': 'Rok', 'vekova_skupina': 'Věk'},
    barmode='group',
    category_orders={"vekova_skupina": ['0–19', '20–44', '45–64', '65–74', '75–84', '85+']},
    color_discrete_sequence=px.colors.sequential.Reds_r,
    template='plotly_white'
)
fig_covid_age.update_layout(hovermode="x unified",)
fig_covid_age.show()

In [36]:
# Seznam všech roků, které jsou v datasetu chřipky jako sloupce
roky_chripka = [str(r) for r in range(2015, 2024)]

# 1. Nejdříve "odpivotujeme" tabulku, aby roky byly v jednom sloupci
df_chripka_age = df_chripka_umrti_abs_kraje.melt(
    id_vars=['Věková kategorie'], 
    value_vars=roky_chripka, 
    var_name='rok', 
    value_name='pocet_umrti'
)

# 2. KLÍČOVÝ KROK: Sečteme počty úmrtí za všechny kraje pro každou kombinaci rok + věk
df_chripka_age_sum = df_chripka_age.groupby(['rok', 'Věková kategorie'])['pocet_umrti'].sum().reset_index()

# 3. Filtrace a úprava typů
df_chripka_age_sum = df_chripka_age_sum[df_chripka_age_sum['Věková kategorie'] != 'Celá populace']
df_chripka_age_sum['rok'] = df_chripka_age_sum['rok'].astype(int)

# 4. Graf pro Chřipku (nyní s celistvými sloupci)
fig_flu_age = px.bar(
    df_chripka_age_sum, 
    x='rok', 
    y='pocet_umrti', 
    color='Věková kategorie',
    title='Chřipka: Vývoj úmrtnosti dle věkových skupin v ČR (2015–2023)',
    labels={'pocet_umrti': 'Počet úmrtí', 'rok': 'Rok', 'Věková kategorie': 'Věk'},
    barmode='group',
    category_orders={"Věková kategorie": ['0–19', '20–44', '45–64', '65–74', '75–84', '85+']},
    color_discrete_sequence=px.colors.sequential.Blues_r,
    template='plotly_white',
)
fig_flu_age.update_layout(hovermode="x unified",)
fig_flu_age.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# COVID-19: Výpočet plně chráněných osob
df_covid_ockov_lidi = df_covid_ockov_kraje.copy()
df_covid_ockov_lidi['datum_dt'] = pd.to_datetime(df_covid_ockov_lidi['datum'], dayfirst=True)

# Funkce pro identifikaci dokončeného očkování
def get_protected_persons(row):
    if 'Janssen' in str(row['vakcina']):
        return row['prvnich_davek']
    else:
        return row['druhych_davek']

df_covid_ockov_lidi['chranene_osoby'] = df_covid_ockov_lidi.apply(get_protected_persons, axis=1)

# Agregace na roky
df_covid_vax_rocni = df_covid_ockov_lidi.assign(rok=df_covid_ockov_lidi['datum_dt'].dt.year).groupby('rok').agg({
    'chranene_osoby': 'sum'
}).reset_index()

# Chřipka: Agregace na roky ze sezón
df_chripka_vax_rocni = df_chripka_ockov_vsichni_kraje.copy()
df_chripka_vax_rocni['rok'] = df_chripka_vax_rocni['sezona'].str.split('-').str[1].astype(int)
df_chripka_vax_rocni = df_chripka_vax_rocni.groupby('rok').agg({
    'pocet_vakcinovanych': 'sum',
    'proockovanost_procenta': 'mean'
}).reset_index()

# --- 2. DEFINICE JEDNOTNÉHO STYLU (PRO OČKOVÁNÍ) ---
# Použijeme zelenou barvu pro vakcinaci a černou pro procenta (pokud jsou)
colors_vax = {'objem': 'seagreen', 'procento': 'black'}

def create_vaccination_chart(df, title, count_col, perc_col=None):
    fig = go.Figure()
    
    # Sloupec pro počet očkovaných osob (levá osa Y)
    fig.add_trace(go.Bar(
        x=df['rok'],
        y=df[count_col],
        name='Očkované osoby',
        marker_color=colors_vax['objem'],
        opacity=0.7,
        yaxis='y1'
    ))

    fig.update_layout(
        title=title,
        xaxis_title='Rok',
        yaxis=dict(title='Počet osob (kumulativně za rok)', side='left'),
        yaxis2=dict(title='Proočkovanost (%)', overlaying='y', side='right', range=[0, 100], showgrid=False),
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified"
    )
    return fig

# --- 3. VYKRESLENÍ GRAFŮ ---

# Graf pro COVID-19 (Dokončené očkování)
fig_vax_covid = create_vaccination_chart(df_covid_vax_rocni, 'COVID-19: Roční přehled plně očkovaných osob', 'chranene_osoby')
fig_vax_covid.show()

# Graf pro Chřipku (Historie očkování)
fig_vax_chripka = create_vaccination_chart(df_chripka_vax_rocni, 'Chřipka: Roční přehled očkovaných osob (Historie)', 'pocet_vakcinovanych', 'proockovanost_procenta')
fig_vax_chripka.show()